# Benchmarking Machine-Learning Models for Density Prediction in Si–O and Si–Al–O Materials

## Project summary

This notebook benchmarks regression models for predicting Materials Project density values using a deliberately small set of physically interpretable composition and structural descriptors. It compares a mean-prediction baseline, composition-focused features, structure-focused features, combined features, and a focused set of linear and nonlinear regressors.

> **Execution status:** Outputs are intentionally cleared. Run every code cell in order to generate counts, plots, metrics, error tables, and the numerical conclusion. Code cells are tagged `run-required`.
>
> **SECURITY ACTION REQUIRED:** An API key was embedded in an earlier version of this notebook. Revoke that key, create a replacement, and store the replacement only in the `MP_API_KEY` environment variable. Never commit credentials, `.env` files, or notebook outputs containing secrets.

## 1. Research Question and Motivation

**Research question**

> How accurately can material density be predicted from a small set of physically interpretable composition and structural descriptors, and how much does combining composition and structural information improve over simpler baselines?

Density is fundamentally mass divided by volume. This makes the task useful as a benchmarking and representation exercise: a model should benefit from descriptors connected to atomic mass and atomic packing, but good performance would not imply that machine learning replaces the physical definition of density.

The analysis asks whether learned models outperform the training-set mean, whether composition or structure is more informative, whether the combined representation helps, and whether nonlinear complexity is justified.

## 2. Data Source and Dataset Definition

Data come from the Materials Project summary endpoint through the official `mp-api` client. The dataset is restricted to non-deprecated entries in the exact Si–O and Si–Al–O chemical systems. The API canonicalizes the ternary query as `Al-O-Si`, while this notebook displays it as `Si-Al-O`.

Only fields needed for identifiers, density, unit-cell size, composition, and symmetry are requested. The notebook records the retrieval date and attempts to record the Materials Project database version. It first looks for `data/materials_snapshot.csv`; when the cache is unavailable, it uses `MP_API_KEY`.

## 3. Imports and Reproducibility Settings

The random seed is applied to NumPy, grouped train–test splitting, and the optional random-forest model. Learned preprocessing remains inside each scikit-learn pipeline.

In [ ]:
# RUN THIS CELL
import json
import os
import platform
import random
import sys
from datetime import date
from importlib import metadata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from mp_api.client import MPRester
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

RANDOM_SEED = 42
TEST_SIZE = 0.20
USE_CACHED_DATA = True

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
FIGURES_DIR = PROJECT_ROOT / "figures"
CACHE_PATH = DATA_DIR / "materials_snapshot.csv"
CACHE_METADATA_PATH = DATA_DIR / "materials_snapshot_metadata.json"

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(context="notebook", style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Use cached data: {USE_CACHED_DATA}")


## 4. Secure Materials Project Data Retrieval

The retrieval workflow loads the cache when requested and available; otherwise it reads `MP_API_KEY`, fails clearly if the variable is absent, and queries the current Materials Project summary-search interface with a context manager.

The stoichiometrically weighted mean atomic mass is calculated as composition weight divided by atom count. This corrects the earlier unweighted average of unique elemental masses. Volume per atom is unit-cell volume divided by site count and is more comparable than raw unit-cell volume when cells contain different numbers of atoms.

In [ ]:
# RUN THIS CELL
CHEMICAL_SYSTEM_QUERIES = ["Si-O", "Al-O-Si"]
REQUESTED_FIELDS = [
    "material_id",
    "formula_pretty",
    "chemsys",
    "density",
    "volume",
    "nsites",
    "nelements",
    "composition",
    "symmetry",
]


def enum_or_value_to_string(value):
    if value is None:
        return None
    return str(getattr(value, "value", value))


def display_chemical_system(composition, api_chemsys):
    if composition is not None:
        element_symbols = {str(element) for element in composition.elements}
        if element_symbols == {"Si", "O"}:
            return "Si-O"
        if element_symbols == {"Si", "Al", "O"}:
            return "Si-Al-O"
    return str(api_chemsys) if api_chemsys is not None else None


def summary_document_to_record(document):
    composition = getattr(document, "composition", None)
    symmetry = getattr(document, "symmetry", None)
    volume = getattr(document, "volume", None)
    number_of_sites = getattr(document, "nsites", None)

    formula = getattr(document, "formula_pretty", None)
    if not formula and composition is not None:
        formula = composition.reduced_formula

    mean_atomic_mass = None
    if composition is not None and composition.num_atoms:
        mean_atomic_mass = float(composition.weight / composition.num_atoms)

    volume_per_atom = None
    if volume is not None and number_of_sites:
        volume_per_atom = float(volume / number_of_sites)

    return {
        "Material ID": str(document.material_id),
        "Formula": str(formula) if formula is not None else None,
        "Chemical System": display_chemical_system(
            composition, getattr(document, "chemsys", None)
        ),
        "Density": float(document.density) if document.density is not None else None,
        "Volume": float(volume) if volume is not None else None,
        "Number of Sites": int(number_of_sites) if number_of_sites is not None else None,
        "Number of Elements": (
            int(document.nelements) if document.nelements is not None else None
        ),
        "Crystal System": (
            enum_or_value_to_string(getattr(symmetry, "crystal_system", None))
            if symmetry is not None
            else None
        ),
        "Space Group Number": (
            int(symmetry.number)
            if symmetry is not None and symmetry.number is not None
            else None
        ),
        "Mean Atomic Mass": mean_atomic_mass,
        "Volume per Atom": volume_per_atom,
    }


In [ ]:
# RUN THIS CELL
loaded_from_cache = USE_CACHED_DATA and CACHE_PATH.exists()

if loaded_from_cache:
    materials_raw_df = pd.read_csv(CACHE_PATH)
    if CACHE_METADATA_PATH.exists():
        retrieval_metadata = json.loads(
            CACHE_METADATA_PATH.read_text(encoding="utf-8")
        )
    else:
        retrieval_metadata = {
            "data_retrieval_date": "Unknown (cache metadata unavailable)",
            "materials_project_database_version": "Unknown (cache metadata unavailable)",
            "source": "Cached Materials Project snapshot",
        }
    print(f"Loaded cached dataset: {CACHE_PATH}")
else:
    if USE_CACHED_DATA:
        print(
            f"Cached dataset not found at {CACHE_PATH}. "
            "Falling back to the Materials Project API."
        )

    api_key = os.getenv("MP_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "MP_API_KEY is not set. Add the Materials Project API key "
            "as an environment variable before running this notebook."
        )

    retrieval_date = date.today().isoformat()
    database_version = "Unavailable from client"

    with MPRester(api_key) as mpr:
        try:
            database_version = str(mpr.get_database_version())
        except Exception as exc:
            print(
                "Materials Project database version could not be recorded: "
                f"{type(exc).__name__}: {exc}"
            )

        documents = mpr.materials.summary.search(
            chemsys=CHEMICAL_SYSTEM_QUERIES,
            deprecated=False,
            include_gnome=False,
            fields=REQUESTED_FIELDS,
        )

    materials_raw_df = pd.DataFrame(
        summary_document_to_record(document) for document in documents
    )
    retrieval_metadata = {
        "data_retrieval_date": retrieval_date,
        "materials_project_database_version": database_version,
        "source": "Materials Project summary API",
        "chemical_system_queries": CHEMICAL_SYSTEM_QUERIES,
        "include_gnome": False,
        "deprecated": False,
    }

print(f"Records available before validation: {len(materials_raw_df):,}")
display(materials_raw_df.head())
print(json.dumps(retrieval_metadata, indent=2))


## 5. Data Cleaning and Quality Checks

No blanket `dropna()` call is used. Before removing records, the notebook reports the initial row count, data types, missing values, duplicate material IDs, nonfinite numeric values, and nonpositive physical values. Missing crystal-system labels are retained as the explicit category `Unknown`. Space-group number is kept for descriptive identification only.

In [ ]:
# RUN THIS CELL
EXPECTED_COLUMNS = [
    "Material ID",
    "Formula",
    "Chemical System",
    "Density",
    "Volume",
    "Number of Sites",
    "Number of Elements",
    "Crystal System",
    "Space Group Number",
    "Mean Atomic Mass",
    "Volume per Atom",
]

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(materials_raw_df.columns))
if missing_columns:
    raise KeyError(
        "The loaded dataset is missing required columns: "
        + ", ".join(missing_columns)
    )

working_df = materials_raw_df[EXPECTED_COLUMNS].copy()
initial_record_count = len(working_df)

print(f"Initial number of records: {initial_record_count:,}")
print("\nInitial data types:")
display(working_df.dtypes.rename("dtype").to_frame())

print("\nMissing values by column before type conversion:")
display(working_df.isna().sum().rename("Missing Values").to_frame())

duplicate_material_ids = working_df.duplicated(
    subset="Material ID", keep=False
)
print(
    "\nRows participating in duplicate Material IDs: "
    f"{int(duplicate_material_ids.sum()):,}"
)

for column in ["Material ID", "Formula", "Chemical System", "Crystal System"]:
    working_df[column] = working_df[column].replace(
        r"^\s*$", np.nan, regex=True
    )

numeric_columns = [
    "Density",
    "Volume",
    "Number of Sites",
    "Number of Elements",
    "Space Group Number",
    "Mean Atomic Mass",
    "Volume per Atom",
]
for column in numeric_columns:
    working_df[column] = pd.to_numeric(
        working_df[column], errors="coerce"
    )

nonfinite_counts = (
    ~np.isfinite(working_df[numeric_columns])
).sum().rename("Nonfinite Values")
print("\nInvalid or nonfinite numeric values by column:")
display(nonfinite_counts.to_frame())

nonpositive_counts = pd.Series(
    {
        "Density <= 0": int((working_df["Density"] <= 0).sum()),
        "Volume <= 0": int((working_df["Volume"] <= 0).sum()),
        "Number of Sites <= 0": int(
            (working_df["Number of Sites"] <= 0).sum()
        ),
        "Volume per Atom <= 0": int(
            (working_df["Volume per Atom"] <= 0).sum()
        ),
    },
    name="Count",
)
print("\nZero or negative values:")
display(nonpositive_counts.to_frame())


In [ ]:
# RUN THIS CELL
REQUIRED_FOR_ANALYSIS = [
    "Material ID",
    "Formula",
    "Chemical System",
    "Density",
    "Volume",
    "Number of Sites",
    "Number of Elements",
    "Mean Atomic Mass",
    "Volume per Atom",
]
required_numeric_columns = [
    "Density",
    "Volume",
    "Number of Sites",
    "Number of Elements",
    "Mean Atomic Mass",
    "Volume per Atom",
]

reason_masks = {
    "Duplicate Material ID after first occurrence": working_df.duplicated(
        subset="Material ID", keep="first"
    ),
    "Missing required analysis value": working_df[
        REQUIRED_FOR_ANALYSIS
    ].isna().any(axis=1),
    "Nonfinite required numeric value": (
        ~np.isfinite(working_df[required_numeric_columns])
    ).any(axis=1),
    "Density is zero or negative": working_df["Density"] <= 0,
    "Volume is zero or negative": working_df["Volume"] <= 0,
    "Number of Sites is zero or negative": (
        working_df["Number of Sites"] <= 0
    ),
    "Number of Elements is zero or negative": (
        working_df["Number of Elements"] <= 0
    ),
    "Mean Atomic Mass is zero or negative": (
        working_df["Mean Atomic Mass"] <= 0
    ),
    "Volume per Atom is zero or negative": (
        working_df["Volume per Atom"] <= 0
    ),
}

removal_summary = pd.Series(
    {reason: int(mask.sum()) for reason, mask in reason_masks.items()},
    name="Flagged Rows",
)
display(removal_summary.to_frame())

invalid_row_mask = pd.Series(False, index=working_df.index)
for mask in reason_masks.values():
    invalid_row_mask |= mask.fillna(True)

removed_record_count = int(invalid_row_mask.sum())
materials_df = working_df.loc[~invalid_row_mask].copy()

materials_df["Crystal System"] = (
    materials_df["Crystal System"].fillna("Unknown").astype(str)
)
materials_df["Space Group Number"] = (
    materials_df["Space Group Number"].round().astype("Int64")
)
materials_df["Number of Sites"] = (
    materials_df["Number of Sites"].round().astype(int)
)
materials_df["Number of Elements"] = (
    materials_df["Number of Elements"].round().astype(int)
)
materials_df = materials_df.sort_values(
    "Material ID"
).reset_index(drop=True)

print(f"Records removed: {removed_record_count:,}")
print(f"Records retained: {len(materials_df):,}")
display(materials_df.dtypes.rename("dtype").to_frame())

if materials_df.empty:
    raise ValueError("No valid records remain after cleaning.")

materials_df.to_csv(CACHE_PATH, index=False)
CACHE_METADATA_PATH.write_text(
    json.dumps(retrieval_metadata, indent=2),
    encoding="utf-8",
)
print(f"Saved reproducible snapshot: {CACHE_PATH}")
print(f"Saved snapshot metadata: {CACHE_METADATA_PATH}")
display(materials_df.head())


The removal-summary flags can overlap, so their sum may exceed the number of unique removed rows. Review unexpectedly large counts before continuing. The saved snapshot contains no API credential and supports reproducible modeling without a fresh API request.

## 6. Exploratory Data Analysis

The exploratory analysis describes distributions and associations without making causal claims. Space-group number is excluded from the correlation matrix because its numbering is not a continuous physical magnitude. `Number of Elements` should be interpreted carefully because it may mainly separate binary Si–O from ternary Si–Al–O entries.

In [ ]:
# RUN THIS CELL
print(
    f"Dataset dimensions: {materials_df.shape[0]:,} rows × "
    f"{materials_df.shape[1]:,} columns"
)
print(f"Unique formulas: {materials_df['Formula'].nunique():,}")
print(
    f"Unique material IDs: "
    f"{materials_df['Material ID'].nunique():,}"
)

summary_columns = [
    "Density",
    "Volume",
    "Number of Sites",
    "Number of Elements",
    "Mean Atomic Mass",
    "Volume per Atom",
]
display(materials_df[summary_columns].describe().T)


In [ ]:
# RUN THIS CELL
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(
    data=materials_df,
    x="Density",
    hue="Chemical System",
    bins="auto",
    element="step",
    common_norm=False,
    ax=ax,
)
ax.set_title("Density Distribution by Chemical System")
ax.set_xlabel(r"Density (g cm$^{-3}$)")
ax.set_ylabel("Number of Materials")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "density_distribution.png", dpi=300)
plt.show()


In [ ]:
# RUN THIS CELL
chemical_system_counts = (
    materials_df["Chemical System"]
    .value_counts(dropna=False)
    .rename_axis("Chemical System")
    .rename("Number of Materials")
)
crystal_system_counts = (
    materials_df["Crystal System"]
    .value_counts(dropna=False)
    .rename_axis("Crystal System")
    .rename("Number of Materials")
)

print("Materials by chemical system:")
display(chemical_system_counts.to_frame())
print("Materials by crystal system:")
display(crystal_system_counts.to_frame())


In [ ]:
# RUN THIS CELL
correlation_columns = [
    "Density",
    "Volume",
    "Number of Sites",
    "Number of Elements",
    "Mean Atomic Mass",
    "Volume per Atom",
]
correlation_matrix = materials_df[correlation_columns].corr(
    method="pearson"
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    square=True,
    cmap="vlag",
    center=0,
    ax=ax,
)
ax.set_title("Pearson Correlation Matrix for Numeric Variables")
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "numeric_correlation_heatmap.png", dpi=300
)
plt.show()


In [ ]:
# RUN THIS CELL
scatter_descriptors = [
    ("Mean Atomic Mass", r"Mean Atomic Mass (u atom$^{-1}$)"),
    (
        "Volume per Atom",
        r"Volume per Atom ($\AA^3$ atom$^{-1}$)",
    ),
]

for descriptor, x_label in scatter_descriptors:
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.scatterplot(
        data=materials_df,
        x=descriptor,
        y="Density",
        hue="Chemical System",
        style="Crystal System",
        alpha=0.75,
        ax=ax,
    )
    ax.set_title(f"Density versus {descriptor}")
    ax.set_xlabel(x_label)
    ax.set_ylabel(r"Density (g cm$^{-3}$)")
    fig.tight_layout()
    safe_name = descriptor.lower().replace(" ", "_")
    fig.savefig(
        FIGURES_DIR / f"density_vs_{safe_name}.png",
        dpi=300,
    )
    plt.show()


### Exploratory-analysis interpretation

Density is mathematically constrained by mass and volume, so associations with mean atomic mass and volume per atom are physically expected. Correlation alone does not prove causation.

**Dataset-specific interpretation:** `[GENERATED AFTER RUNNING THE NOTEBOOK]`

## 7. Physics-Informed Feature Engineering

Three representations are evaluated:

- **Composition:** Mean Atomic Mass, Number of Elements, and categorical Chemical System.
- **Structure:** Volume per Atom and categorical Crystal System.
- **Combined:** all selected composition and structure descriptors.

Raw unit-cell volume is excluded because it is extensive and depends on cell size. Number of sites is retained for quality checks but excluded as a predictor because it can reflect crystallographic cell choice. Space-group number is not used as a continuous input.

The earlier unused PCA calculation has been removed. With a small, interpretable descriptor set, unevaluated dimensionality reduction would add complexity without scientific benefit.

In [ ]:
# RUN THIS CELL
FEATURE_SETS = {
    "Composition": {
        "numeric": ["Mean Atomic Mass", "Number of Elements"],
        "categorical": ["Chemical System"],
    },
    "Structure": {
        "numeric": ["Volume per Atom"],
        "categorical": ["Crystal System"],
    },
    "Combined": {
        "numeric": [
            "Mean Atomic Mass",
            "Number of Elements",
            "Volume per Atom",
        ],
        "categorical": ["Chemical System", "Crystal System"],
    },
}

for feature_set_name, feature_specification in FEATURE_SETS.items():
    selected_columns = (
        feature_specification["numeric"]
        + feature_specification["categorical"]
    )
    print(f"{feature_set_name}: {selected_columns}")


## 8. Train–Test Splitting Strategy

A random row-level split can place polymorphs or related entries with the same reduced formula in both training and test sets. This notebook uses `GroupShuffleSplit` with reduced formula as the group. Each formula is assigned wholly to training or test data, producing a more demanding estimate for unfamiliar formulas and reducing leakage.

The notebook stops explicitly if there are too few formula groups; it does not silently fall back to a row-level split.

In [ ]:
# RUN THIS CELL
formula_groups = materials_df["Formula"].astype(str)
unique_formula_count = formula_groups.nunique()
print(f"Unique formula groups: {unique_formula_count:,}")

if unique_formula_count < 5:
    raise ValueError(
        "Fewer than five unique formula groups are available. "
        "A grouped train-test evaluation would be unreliable."
    )

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
)
train_indices, test_indices = next(
    group_splitter.split(materials_df, groups=formula_groups)
)

train_df = materials_df.iloc[train_indices].copy().reset_index(drop=True)
test_df = materials_df.iloc[test_indices].copy().reset_index(drop=True)
train_groups = train_df["Formula"].astype(str)
test_groups = test_df["Formula"].astype(str)

group_overlap = set(train_groups).intersection(set(test_groups))
if group_overlap:
    raise RuntimeError(
        "Formula-group leakage detected between training and test sets."
    )
if len(test_df) < 2:
    raise ValueError(
        "The grouped test split contains fewer than two rows; "
        "R² cannot be evaluated reliably."
    )

print(f"Training rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Training formula groups: {train_groups.nunique():,}")
print(f"Test formula groups: {test_groups.nunique():,}")
print(f"Formula groups shared across splits: {len(group_overlap)}")

display(
    pd.DataFrame(
        {
            "Split": ["Training", "Test"],
            "Rows": [len(train_df), len(test_df)],
            "Unique Formulas": [
                train_groups.nunique(),
                test_groups.nunique(),
            ],
        }
    )
)


## 9. Preprocessing and Modeling Pipelines

All models receive raw pandas columns. Numeric features are standardized; categorical features are one-hot encoded with unknown categories ignored. No imputation occurs in the model pipeline because unusable rows were explicitly reported and removed during cleaning.

The model set is focused: mean baseline, linear regression, ridge regression, polynomial ridge, KNN, and a random forest only when the training data are sufficiently large. A neural network is omitted because this compact tabular problem does not by itself justify the added optimization complexity.

In [ ]:
# RUN THIS CELL
def make_preprocessor(feature_specification):
    transformers = []
    numeric_features = feature_specification["numeric"]
    categorical_features = feature_specification["categorical"]

    if numeric_features:
        numeric_pipeline = Pipeline(
            steps=[("scaler", StandardScaler())]
        )
        transformers.append(
            ("numeric", numeric_pipeline, numeric_features)
        )

    if categorical_features:
        categorical_pipeline = Pipeline(
            steps=[
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        drop="first",
                        sparse_output=False,
                    ),
                )
            ]
        )
        transformers.append(
            ("categorical", categorical_pipeline, categorical_features)
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )


def make_model_pipeline(feature_specification, model_name):
    preprocessor = make_preprocessor(feature_specification)

    if model_name == "Mean Baseline":
        model_steps = [
            ("preprocessor", preprocessor),
            ("model", DummyRegressor(strategy="mean")),
        ]
    elif model_name == "Linear Regression":
        model_steps = [
            ("preprocessor", preprocessor),
            ("model", LinearRegression()),
        ]
    elif model_name == "Ridge Regression":
        model_steps = [
            ("preprocessor", preprocessor),
            ("model", Ridge()),
        ]
    elif model_name == "Polynomial Ridge":
        model_steps = [
            ("preprocessor", preprocessor),
            (
                "polynomial",
                PolynomialFeatures(include_bias=False),
            ),
            ("polynomial_scaler", StandardScaler()),
            ("model", Ridge()),
        ]
    elif model_name == "K-Nearest Neighbors":
        model_steps = [
            ("preprocessor", preprocessor),
            ("model", KNeighborsRegressor()),
        ]
    elif model_name == "Random Forest":
        model_steps = [
            ("preprocessor", preprocessor),
            (
                "model",
                RandomForestRegressor(
                    random_state=RANDOM_SEED,
                    n_jobs=-1,
                ),
            ),
        ]
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    return Pipeline(steps=model_steps)


## 10. Cross-Validation and Hyperparameter Selection

Selection uses only the training partition. `GroupKFold` keeps identical formula groups within one fold. Search spaces are deliberately small: ridge alpha, polynomial degree 2–3, a limited KNN grid, and a compact random-forest grid if supported. The primary metric is grouped cross-validation RMSE, reported as mean and standard deviation.

In [ ]:
# RUN THIS CELL
training_group_count = train_groups.nunique()
cross_validation_splits = min(5, training_group_count)

if cross_validation_splits < 3:
    raise ValueError(
        "Fewer than three training formula groups are available for "
        "grouped cross-validation."
    )

group_cross_validator = GroupKFold(
    n_splits=cross_validation_splits
)

cv_split_indices = list(
    group_cross_validator.split(
        train_df,
        groups=train_groups,
    )
)
estimated_smallest_fold_training_size = min(
    len(fold_train_indices)
    for fold_train_indices, _ in cv_split_indices
)
knn_neighbor_candidates = [
    neighbor_count
    for neighbor_count in [3, 5, 7, 9, 11]
    if neighbor_count <= estimated_smallest_fold_training_size
]
if not knn_neighbor_candidates:
    knn_neighbor_candidates = [1]

include_random_forest = (
    len(train_df) >= 100 and training_group_count >= 20
)

MODEL_PARAMETER_GRIDS = {
    "Linear Regression": [{}],
    "Ridge Regression": {
        "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
    },
    "Polynomial Ridge": {
        "polynomial__degree": [2, 3],
        "model__alpha": [0.1, 1.0, 10.0],
    },
    "K-Nearest Neighbors": {
        "model__n_neighbors": knn_neighbor_candidates,
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2],
    },
}

if include_random_forest:
    MODEL_PARAMETER_GRIDS["Random Forest"] = {
        "model__n_estimators": [200],
        "model__max_depth": [None, 5, 10],
        "model__min_samples_leaf": [1, 2],
    }

print(f"Grouped CV folds: {cross_validation_splits}")
print(f"KNN neighbor candidates: {knn_neighbor_candidates}")
print(f"Include random forest: {include_random_forest}")
print("Models to evaluate:", list(MODEL_PARAMETER_GRIDS))


In [ ]:
# RUN THIS CELL
target_column = "Density"
training_target = train_df[target_column]
test_target = test_df[target_column]

search_records = []
fitted_searches = {}

baseline_specification = FEATURE_SETS["Combined"]
baseline_columns = (
    baseline_specification["numeric"]
    + baseline_specification["categorical"]
)
baseline_search = GridSearchCV(
    estimator=make_model_pipeline(
        baseline_specification, "Mean Baseline"
    ),
    param_grid=[{}],
    scoring="neg_root_mean_squared_error",
    cv=group_cross_validator,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
    error_score="raise",
)
baseline_search.fit(
    train_df[baseline_columns],
    training_target,
    groups=train_groups,
)

baseline_key = ("Baseline", "Mean Baseline")
fitted_searches[baseline_key] = {
    "search": baseline_search,
    "columns": baseline_columns,
}
search_records.append(
    {
        "Feature Set": "Baseline",
        "Model": "Mean Baseline",
        "Cross-Validation RMSE Mean": -baseline_search.best_score_,
        "Cross-Validation RMSE Standard Deviation": (
            baseline_search.cv_results_["std_test_score"][
                baseline_search.best_index_
            ]
        ),
        "Best Parameters": baseline_search.best_params_,
    }
)

for feature_set_name, feature_specification in FEATURE_SETS.items():
    feature_columns = (
        feature_specification["numeric"]
        + feature_specification["categorical"]
    )
    for model_name, parameter_grid in MODEL_PARAMETER_GRIDS.items():
        search = GridSearchCV(
            estimator=make_model_pipeline(
                feature_specification, model_name
            ),
            param_grid=parameter_grid,
            scoring="neg_root_mean_squared_error",
            cv=group_cross_validator,
            n_jobs=-1,
            refit=True,
            return_train_score=False,
            error_score="raise",
        )
        search.fit(
            train_df[feature_columns],
            training_target,
            groups=train_groups,
        )

        model_key = (feature_set_name, model_name)
        fitted_searches[model_key] = {
            "search": search,
            "columns": feature_columns,
        }
        search_records.append(
            {
                "Feature Set": feature_set_name,
                "Model": model_name,
                "Cross-Validation RMSE Mean": -search.best_score_,
                "Cross-Validation RMSE Standard Deviation": (
                    search.cv_results_["std_test_score"][
                        search.best_index_
                    ]
                ),
                "Best Parameters": search.best_params_,
            }
        )

cross_validation_results_df = (
    pd.DataFrame(search_records)
    .sort_values(
        [
            "Cross-Validation RMSE Mean",
            "Cross-Validation RMSE Standard Deviation",
        ]
    )
    .reset_index(drop=True)
)

display(
    cross_validation_results_df.style.format(
        {
            "Cross-Validation RMSE Mean": "{:.4f}",
            "Cross-Validation RMSE Standard Deviation": "{:.4f}",
        }
    )
)


### Model selection before test evaluation

The next cell selects the candidate with the lowest grouped cross-validation RMSE before any held-out test metric is calculated.

In [ ]:
# RUN THIS CELL
selected_cv_row = cross_validation_results_df.iloc[0]
selected_model_key = (
    selected_cv_row["Feature Set"],
    selected_cv_row["Model"],
)

print("Selected using grouped cross-validation only:")
print(f"Feature set: {selected_model_key[0]}")
print(f"Model: {selected_model_key[1]}")
print(
    "CV RMSE: "
    f"{selected_cv_row['Cross-Validation RMSE Mean']:.4f} ± "
    f"{selected_cv_row['Cross-Validation RMSE Standard Deviation']:.4f}"
)


## 11. Final Test-Set Evaluation

Each finalized, cross-validated pipeline is evaluated once on the untouched grouped test set. RMSE is reported in density units, alongside MAE and R². The final model remains the one selected by cross-validation even if another candidate happens to have a lower test RMSE.

In [ ]:
# RUN THIS CELL
test_evaluation_records = []
test_predictions = {}

for model_key, fitted_information in fitted_searches.items():
    feature_set_name, model_name = model_key
    fitted_search = fitted_information["search"]
    feature_columns = fitted_information["columns"]

    predictions = fitted_search.best_estimator_.predict(
        test_df[feature_columns]
    )
    test_predictions[model_key] = predictions

    test_evaluation_records.append(
        {
            "Feature Set": feature_set_name,
            "Model": model_name,
            "Test RMSE": (
                mean_squared_error(test_target, predictions) ** 0.5
            ),
            "Test MAE": mean_absolute_error(
                test_target, predictions
            ),
            "Test R²": r2_score(test_target, predictions),
        }
    )

test_evaluation_df = pd.DataFrame(test_evaluation_records)
model_comparison_df = (
    cross_validation_results_df.merge(
        test_evaluation_df,
        on=["Feature Set", "Model"],
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        ["Cross-Validation RMSE Mean", "Test RMSE"]
    )
    .reset_index(drop=True)
)

comparison_columns = [
    "Feature Set",
    "Model",
    "Cross-Validation RMSE Mean",
    "Cross-Validation RMSE Standard Deviation",
    "Test RMSE",
    "Test MAE",
    "Test R²",
    "Best Parameters",
]
display(
    model_comparison_df[comparison_columns].style.format(
        {
            "Cross-Validation RMSE Mean": "{:.4f}",
            "Cross-Validation RMSE Standard Deviation": "{:.4f}",
            "Test RMSE": "{:.4f}",
            "Test MAE": "{:.4f}",
            "Test R²": "{:.4f}",
        }
    )
)


## 12. Model Comparison

The chart compares the one-time test RMSE for all finalized candidates. Selection should still be interpreted using grouped cross-validation rather than by choosing the smallest test bar after inspection.

In [ ]:
# RUN THIS CELL
plotting_df = model_comparison_df.copy()
plotting_df["Candidate"] = (
    plotting_df["Feature Set"] + " — " + plotting_df["Model"]
)
plotting_df = plotting_df.sort_values(
    "Test RMSE", ascending=False
)

fig_height = max(6, 0.38 * len(plotting_df))
fig, ax = plt.subplots(figsize=(10, fig_height))
ax.barh(
    plotting_df["Candidate"],
    plotting_df["Test RMSE"],
)
ax.set_title("One-Time Held-Out Test RMSE by Candidate")
ax.set_xlabel(r"Test RMSE (g cm$^{-3}$)")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "model_comparison.png", dpi=300
)
plt.show()


In [ ]:
# RUN THIS CELL
selected_search_information = fitted_searches[
    selected_model_key
]
selected_estimator = (
    selected_search_information["search"].best_estimator_
)
selected_feature_columns = (
    selected_search_information["columns"]
)
selected_test_predictions = test_predictions[
    selected_model_key
]

selected_test_row = model_comparison_df[
    (
        model_comparison_df["Feature Set"]
        == selected_model_key[0]
    )
    & (
        model_comparison_df["Model"]
        == selected_model_key[1]
    )
].iloc[0]

print("Final model selected by grouped cross-validation:")
display(selected_test_row[comparison_columns].to_frame("Value"))


## 13. Residual and Error Analysis

The parity and residual plots focus only on the cross-validation-selected model. Residual is defined as actual density minus predicted density. Positive residuals indicate underprediction; negative residuals indicate overprediction.

In [ ]:
# RUN THIS CELL
actual_values = test_target.to_numpy()
predicted_values = np.asarray(
    selected_test_predictions
)

axis_min = min(
    actual_values.min(), predicted_values.min()
)
axis_max = max(
    actual_values.max(), predicted_values.max()
)
axis_range = axis_max - axis_min
axis_padding = (
    0.05 * axis_range if axis_range > 0 else 0.1
)
axis_limits = (
    axis_min - axis_padding,
    axis_max + axis_padding,
)

fig, ax = plt.subplots(figsize=(6.5, 6.5))
sns.scatterplot(
    x=actual_values,
    y=predicted_values,
    hue=test_df["Chemical System"],
    style=test_df["Crystal System"],
    s=70,
    alpha=0.80,
    ax=ax,
)
ax.plot(
    axis_limits,
    axis_limits,
    linestyle="--",
    linewidth=1.5,
    label="Ideal prediction",
)
ax.set_xlim(axis_limits)
ax.set_ylim(axis_limits)
ax.set_aspect("equal", adjustable="box")
ax.set_title(
    f"Parity Plot: {selected_model_key[0]} / "
    f"{selected_model_key[1]}"
)
ax.set_xlabel(r"Actual Density (g cm$^{-3}$)")
ax.set_ylabel(r"Predicted Density (g cm$^{-3}$)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "parity_plot.png", dpi=300)
plt.show()


In [ ]:
# RUN THIS CELL
selected_residuals = (
    actual_values - predicted_values
)

fig, ax = plt.subplots(figsize=(7.5, 5))
sns.scatterplot(
    x=predicted_values,
    y=selected_residuals,
    hue=test_df["Chemical System"],
    style=test_df["Crystal System"],
    s=70,
    alpha=0.80,
    ax=ax,
)
ax.axhline(0.0, linestyle="--", linewidth=1.5)
ax.set_title(
    f"Residuals: {selected_model_key[0]} / "
    f"{selected_model_key[1]}"
)
ax.set_xlabel(r"Predicted Density (g cm$^{-3}$)")
ax.set_ylabel(
    r"Residual: Actual − Predicted (g cm$^{-3}$)"
)
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "residual_vs_predicted.png",
    dpi=300,
)
plt.show()


In [ ]:
# RUN THIS CELL
error_analysis_df = test_df[
    [
        "Material ID",
        "Formula",
        "Chemical System",
        "Crystal System",
        "Volume per Atom",
        "Density",
    ]
].copy()

error_analysis_df = error_analysis_df.rename(
    columns={"Density": "Actual Density"}
)
error_analysis_df["Predicted Density"] = (
    predicted_values
)
error_analysis_df["Residual"] = (
    error_analysis_df["Actual Density"]
    - error_analysis_df["Predicted Density"]
)
error_analysis_df["Absolute Error"] = (
    error_analysis_df["Residual"].abs()
)

largest_errors_df = (
    error_analysis_df.sort_values(
        "Absolute Error", ascending=False
    )
    .head(min(10, len(error_analysis_df)))
    .reset_index(drop=True)
)

required_error_columns = [
    "Material ID",
    "Formula",
    "Chemical System",
    "Actual Density",
    "Predicted Density",
    "Residual",
    "Absolute Error",
]
display(
    largest_errors_df[
        required_error_columns
    ].style.format(
        {
            "Actual Density": "{:.4f}",
            "Predicted Density": "{:.4f}",
            "Residual": "{:.4f}",
            "Absolute Error": "{:.4f}",
        }
    )
)


In [ ]:
# RUN THIS CELL
chemical_system_error_summary = (
    error_analysis_df.groupby(
        "Chemical System", dropna=False
    )
    .agg(
        Materials=("Material ID", "count"),
        Mean_Absolute_Error=(
            "Absolute Error", "mean"
        ),
        Median_Absolute_Error=(
            "Absolute Error", "median"
        ),
    )
    .sort_values(
        "Mean_Absolute_Error", ascending=False
    )
)

crystal_system_error_summary = (
    error_analysis_df.groupby(
        "Crystal System", dropna=False
    )
    .agg(
        Materials=("Material ID", "count"),
        Mean_Absolute_Error=(
            "Absolute Error", "mean"
        ),
        Median_Absolute_Error=(
            "Absolute Error", "median"
        ),
    )
    .sort_values(
        "Mean_Absolute_Error", ascending=False
    )
)

training_vpa_min = train_df[
    "Volume per Atom"
].min()
training_vpa_max = train_df[
    "Volume per Atom"
].max()
largest_errors_outside_training_vpa = (
    (
        largest_errors_df["Volume per Atom"]
        < training_vpa_min
    )
    | (
        largest_errors_df["Volume per Atom"]
        > training_vpa_max
    )
).sum()

print("Error summary by chemical system:")
display(chemical_system_error_summary)
print("Error summary by crystal system:")
display(crystal_system_error_summary)
print(
    "Largest-error entries outside the training "
    "Volume-per-Atom range: "
    f"{int(largest_errors_outside_training_vpa)} of "
    f"{len(largest_errors_df)}"
)
print(
    "All test formulas are absent from training "
    "by construction."
)


In [ ]:
# RUN THIS CELL
fig, ax = plt.subplots(figsize=(7.5, 5))
sns.boxplot(
    data=error_analysis_df,
    x="Chemical System",
    y="Residual",
    ax=ax,
)
sns.stripplot(
    data=error_analysis_df,
    x="Chemical System",
    y="Residual",
    alpha=0.60,
    ax=ax,
)
ax.axhline(0.0, linestyle="--", linewidth=1.5)
ax.set_title(
    "Residual Distribution by Chemical System"
)
ax.set_xlabel("Chemical System")
ax.set_ylabel(r"Residual (g cm$^{-3}$)")
fig.tight_layout()
fig.savefig(
    FIGURES_DIR
    / "residuals_by_chemical_system.png",
    dpi=300,
)
plt.show()


### Error-analysis interpretation guide

Use the generated tables to assess whether large errors are concentrated in one chemical system, one crystal system, unusual density values, or volume-per-atom values outside the training range. Because every test formula is intentionally unfamiliar, formula novelty applies to the whole test set.

Mechanistic explanations should be labeled as hypotheses unless supported by additional structure-level or chemistry-level evidence.

**Observed error pattern:** `[GENERATED AFTER RUNNING THE NOTEBOOK]`

## 14. Limitations

- **Limited chemical diversity:** only Si–O and Si–Al–O systems are included.
- **Related structures:** formula grouping reduces leakage, but entries can still be related through broader structural families.
- **Computational targets:** Materials Project density values are derived from computed structures rather than independent experimental measurements.
- **Dataset size:** grouped evaluation can have substantial variance with few formula groups.
- **Limited descriptors:** local environments, bonding, oxidation states, and energetic information are omitted.
- **Potential imbalance:** chemical systems and crystal systems may be unevenly represented.
- **Restricted generalizability:** results do not establish performance beyond these chemical systems.
- **No external validation:** there is no separate experimental test database.
- **Known physical constraint:** density is strongly constrained by mass and volume.
- **Single grouped holdout:** repeated nested grouped evaluation would provide a fuller uncertainty estimate if the dataset supports it.

## 15. Conclusion

The numerical conclusion below is generated only after fitting and held-out evaluation. It reports whether the selected candidate beat the mean baseline, which feature set was selected, whether combined structural information improved on composition-only features, and how added complexity compared with simpler linear-family candidates.

`[GENERATED AFTER RUNNING THE NOTEBOOK]`

In [ ]:
# RUN THIS CELL
baseline_result = model_comparison_df[
    (
        model_comparison_df["Feature Set"]
        == "Baseline"
    )
    & (
        model_comparison_df["Model"]
        == "Mean Baseline"
    )
].iloc[0]

composition_best = (
    model_comparison_df[
        model_comparison_df["Feature Set"]
        == "Composition"
    ]
    .sort_values("Test RMSE")
    .iloc[0]
)
structure_best = (
    model_comparison_df[
        model_comparison_df["Feature Set"]
        == "Structure"
    ]
    .sort_values("Test RMSE")
    .iloc[0]
)
combined_best = (
    model_comparison_df[
        model_comparison_df["Feature Set"]
        == "Combined"
    ]
    .sort_values("Test RMSE")
    .iloc[0]
)

baseline_difference = (
    baseline_result["Test RMSE"]
    - selected_test_row["Test RMSE"]
)
baseline_percent_improvement = (
    100.0
    * baseline_difference
    / baseline_result["Test RMSE"]
    if baseline_result["Test RMSE"] != 0
    else np.nan
)
combined_minus_composition = (
    combined_best["Test RMSE"]
    - composition_best["Test RMSE"]
)

simple_model_names = [
    "Linear Regression",
    "Ridge Regression",
]
best_simple_cv = (
    cross_validation_results_df[
        cross_validation_results_df[
            "Model"
        ].isin(simple_model_names)
    ]
    .sort_values(
        "Cross-Validation RMSE Mean"
    )
    .iloc[0]
)
complexity_cv_difference = (
    best_simple_cv[
        "Cross-Validation RMSE Mean"
    ]
    - selected_cv_row[
        "Cross-Validation RMSE Mean"
    ]
)

if (
    selected_test_row["Test RMSE"]
    < baseline_result["Test RMSE"]
):
    baseline_statement = (
        "The selected model outperformed the mean "
        "baseline on the grouped test set by "
        f"{baseline_difference:.4f} g cm⁻³ RMSE "
        f"({baseline_percent_improvement:.1f}% "
        "relative reduction)."
    )
else:
    baseline_statement = (
        "The selected model did not outperform the "
        "mean baseline on the grouped test set; "
        f"its RMSE was "
        f"{selected_test_row['Test RMSE']:.4f} "
        "g cm⁻³ versus "
        f"{baseline_result['Test RMSE']:.4f} "
        "g cm⁻³ for the baseline."
    )

if combined_minus_composition < 0:
    structure_statement = (
        "The best combined-feature candidate had "
        "lower test RMSE than the best "
        "composition-only candidate by "
        f"{abs(combined_minus_composition):.4f} "
        "g cm⁻³, consistent with structural "
        "information adding predictive value in "
        "this split."
    )
elif combined_minus_composition > 0:
    structure_statement = (
        "The best combined-feature candidate had "
        "higher test RMSE than the best "
        "composition-only candidate by "
        f"{combined_minus_composition:.4f} "
        "g cm⁻³, so this split does not show a "
        "test-set benefit from the selected "
        "structural features."
    )
else:
    structure_statement = (
        "The best combined and composition-only "
        "candidates had equal test RMSE at the "
        "displayed precision."
    )

nonlinear_models = {
    "Polynomial Ridge",
    "K-Nearest Neighbors",
    "Random Forest",
}
if selected_model_key[1] in nonlinear_models:
    complexity_statement = (
        "The selected nonlinear model improved "
        "grouped CV RMSE over the best "
        "linear-family candidate by "
        f"{complexity_cv_difference:.4f} g cm⁻³. "
        "This gain should be weighed against "
        "reduced interpretability and fold-to-fold "
        "variation."
    )
else:
    complexity_statement = (
        "A linear-family candidate was selected by "
        "grouped cross-validation, so additional "
        "nonlinear complexity was not required."
    )

conclusion_text = f'''
### Generated numerical conclusion

The final candidate selected using grouped
cross-validation was **{selected_model_key[1]}**
with the **{selected_model_key[0]}** feature set.
Its grouped CV RMSE was
**{selected_cv_row["Cross-Validation RMSE Mean"]:.4f}
± {selected_cv_row["Cross-Validation RMSE Standard Deviation"]:.4f}
g cm⁻³**. On the one-time grouped test set, it
obtained RMSE **{selected_test_row["Test RMSE"]:.4f}
g cm⁻³**, MAE **{selected_test_row["Test MAE"]:.4f}
g cm⁻³**, and R²
**{selected_test_row["Test R²"]:.4f}**.

{baseline_statement}

{structure_statement}

For context, the best structure-only candidate
was **{structure_best["Model"]}** with test RMSE
**{structure_best["Test RMSE"]:.4f} g cm⁻³**,
while the best composition-only candidate was
**{composition_best["Model"]}** with test RMSE
**{composition_best["Test RMSE"]:.4f} g cm⁻³**.

{complexity_statement}

These results apply only to the retrieved Si–O
and Si–Al–O entries and this grouped split. They
do not establish generalization to other chemical
systems or to experimental density measurements.
The task is best interpreted as a benchmark of
compact material representations under a known
mass–volume relationship, not as evidence that
machine learning replaces the physical definition
of density.
'''
display(Markdown(conclusion_text))


## 16. Reproducibility Instructions and References

### Running the project

1. Create and activate a clean Python environment.
2. Install packages from `requirements.txt`.
3. Either place `materials_snapshot.csv` and its metadata JSON in `data/`, or set `USE_CACHED_DATA = False` and define `MP_API_KEY`.
4. Launch Jupyter from the repository root and open `notebooks/materials_density_prediction.ipynb`.
5. Use **Kernel → Restart Kernel and Run All Cells**.
6. Review data-quality diagnostics before interpreting model results.
7. Verify that no credential appears in source or output before publishing.

### Setting the API key

macOS/Linux:

```bash
export MP_API_KEY="your_replacement_key"
```

Windows PowerShell:

```powershell
$env:MP_API_KEY="your_replacement_key"
```

### References

- Materials Project API: https://docs.materialsproject.org/downloading-data/using-the-api
- `mp-api` documentation: https://materialsproject.github.io/api/
- scikit-learn grouped validation: https://scikit-learn.org/stable/modules/cross_validation.html
- scikit-learn pipelines: https://scikit-learn.org/stable/modules/compose.html
- Materials Project database versions: https://docs.materialsproject.org/changes/database-versions

In [ ]:
# RUN THIS CELL
packages_to_report = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "mp-api",
    "pymatgen",
    "jupyter",
]

version_rows = []
for package_name in packages_to_report:
    try:
        package_version = metadata.version(
            package_name
        )
    except metadata.PackageNotFoundError:
        package_version = "Not installed"
    version_rows.append(
        {
            "Package": package_name,
            "Version": package_version,
        }
    )

print(f"Python version: {platform.python_version()}")
print(f"Python executable: {sys.executable}")
print(f"Random seed: {RANDOM_SEED}")
print(
    "Data retrieval date: "
    f"{retrieval_metadata.get('data_retrieval_date', 'Unknown')}"
)
print(
    "Materials Project database version: "
    f"{retrieval_metadata.get('materials_project_database_version', 'Unknown')}"
)
display(pd.DataFrame(version_rows))


### Suggested GitHub repository structure

```text
materials-density-ml/
├── README.md
├── LICENSE
├── requirements.txt
├── .gitignore
├── notebooks/
│   └── materials_density_prediction.ipynb
├── data/
│   ├── materials_snapshot.csv
│   └── materials_snapshot_metadata.json
├── figures/
│   ├── density_distribution.png
│   ├── model_comparison.png
│   ├── parity_plot.png
│   └── residual_vs_predicted.png
└── src/
    └── data_collection.py
```

Choose an appropriate license. Before publication, rerun from a clean environment, verify labels and metrics, and replace every generated placeholder in the README with validated output.